#### Python Memory Management
Memory management in Python involves a combination of automatic garbage collection, reference counting, and various internal optimizations to efficiently manage memory allocation and deallocation. Understanding these mechanisms can help developers write more efficient and robust applications.

1. Key Concepts in Python Memory Management
2. Memory Allocation and Deallocation
3. Reference Counting
4. Garbage Collection
5. The gc Module
6. Memory Management Best Practices

#### Reference Counting
Reference counting is the primary method Python uses to manage memory. Each object in Python maintains a count of references pointing to it. When the reference count drops to zero, the memory occupied by the object is deallocated.

In [ ]:
import sys

a=[] ## variable a points to a list object 
## 2 (one reference from 'a' and (temporary reference)one from getrefcount())
print(sys.getrefcount(a)) #It returns the number of references pointing to the object a.
## In simple words : How many variables / places are currently using the same object.

2


In [ ]:
b=a ## Assigning b = a increases the reference count because both variables point to the same object.
print(sys.getrefcount(b))

3


In [3]:
del b
print(sys.getrefcount(a))

2


#### Garbage Collection
Python includes a cyclic garbage collector to handle reference cycles. Reference cycles occur when objects reference each other, preventing their reference counts from reaching zero.
(Sometimes objects keep pointing to each other, so Python can’t tell they are unused.
The cyclic garbage collector steps in and cleans them up.)

In [ ]:
import gc # gc = Garbage Collector module
## automatically free memory,remove objects that are no longer needed
## enable garbage collection
gc.enable() # Turn ON Python’s automatic garbage collection.
#After this: Python actively looks for unused objects,Frees memory automatically when possible

In [5]:
gc.disable()

In [ ]:
gc.collect() # Returns the number of unreachable objects found and cleaned, immediately reclaim unused memory.

38

In [ ]:
### Get garbage collection stats
print(gc.get_stats()) # Returns statistics about garbage collection activity.

[{'collections': 195, 'collected': 1746, 'uncollectable': 0}, {'collections': 17, 'collected': 500, 'uncollectable': 0}, {'collections': 2, 'collected': 66, 'uncollectable': 0}]


In [ ]:
### get unreachable objects
print(gc.garbage) #A list that stores objects which the garbage collector found but could NOT free.(uncollectable objects.)

[]


####  Memory Management Best Practices
1. Use Local Variables: Local variables have a shorter lifespan and are freed sooner than global variables.
2. Avoid Circular References: Circular references can lead to memory leaks if not properly managed.
3. Use Generators: Generators produce items one at a time and only keep one item in memory at a time, making them memory efficient.
4. Explicitly Delete Objects: Use the del statement to delete variables and objects explicitly.
5. Profile Memory Usage: Use memory profiling tools like tracemalloc and memory_profiler to identify memory leaks and optimize memory usage.

In [ ]:
## Handled Circular reference
import gc

class MyObject:
    def __init__(self, name):
        self.name = name
        print(f"Object {self.name} created")

    def __del__(self): ##__del__ is called a destructor, Python calls it automatically when an object is about to be destroyed
        print(f"Object {self.name} deleted")

##Conceptual thing:
## = changes what a name points to; .ref creates a relationship between objects
## obj1 = obj2 ->obj1 now points to the same object as obj2,The original object of obj1 may be destroyed if no other references exist,No circular reference is created

# Create circular reference
obj1 = MyObject("obj1")
obj2 = MyObject("obj2")
obj1.ref = obj2
obj2.ref = obj1

## del removes the variable’s reference to the object,del does NOT delete the object from memory
del obj1 
del obj2

## Manually trigger the garbage collection
gc.collect()

Object obj1 created
Object obj2 created
Object obj1 deleted
Object obj2 deleted


4

In [10]:
## Generators For Memory Efficiency
#Generators allow you to produce items one at a time, using memory efficiently by only keeping one item in memory at a time.

def generate_numbers(n):
    for i in range(n):
        yield i

## using the generator
for num in generate_numbers(100000):
    print(num)
    if num>10:
        break

##STep wise illustration : 
#Step 0 :Python defines the function,No code inside runs now, contain yield so generator function

# STEP 1:
# generate_numbers(100000) is called.Because it contains 'yield', Python creates a generator object.
# No loop code runs yet.

# STEP 2: The for-loop starts and internally calls next() on the generator.
##Internally : 
##gen = generate_numbers(100000)
##num = next(gen)

# STEP 3:
# Generator starts execution:
# i = 0 → yield 0 → pauses here and returns 0 to the loop.

# STEP 4:
# num = 0 is printed.
# Condition (num > 10) is False → loop continues.

# STEP 5:
# Loop stops immediately.(when 11 > 10)
# Generator is abandoned and remaining values are never generated.

0
1
2
3
4
5
6
7
8
9
10
11


In [13]:
## Profiling Memory USage with tracemalloc
## tracemalloc tracks memory allocations in Python and tells you which lines of code are using memory.
import tracemalloc

def create_list():
    return [i for i in range(10000)]

def main():
    tracemalloc.start()
    
    create_list()
    # take_snapshot() captures current memory usage 
    # statistics('lineno') shows which lines of code used how much memory.
    snapshot = tracemalloc.take_snapshot()
    top_stats = snapshot.statistics('lineno') #statistics() puts the lines using the MOST memory at the TOP of the list.
    ## top_stats is a list
    
    print("[ Top 10 ]")
    for stat in top_stats[::]:
        print(stat)

## eg : example.py:5: size=390 KiB, count=10000, average=40 B
## example.py → the file where memory was allocated
## 5 → the exact line number in that file
## size=390 KiB → total memory used by that line
## count=10000 → number of memory allocations made at that line
## average=40 B → average memory used per allocation


In [14]:
main()

[ Top 10 ]
c:\Users\Dell\anaconda3\envs\venv\lib\selectors.py:315: size=144 KiB, count=3, average=48.0 KiB
c:\Users\Dell\anaconda3\envs\venv\lib\json\decoder.py:353: size=5265 B, count=42, average=125 B
c:\Users\Dell\anaconda3\envs\venv\lib\site-packages\IPython\core\compilerop.py:174: size=2906 B, count=27, average=108 B
c:\Users\Dell\anaconda3\envs\venv\lib\site-packages\IPython\core\history.py:836: size=2380 B, count=7, average=340 B
c:\Users\Dell\anaconda3\envs\venv\lib\site-packages\IPython\core\history.py:782: size=2135 B, count=2, average=1068 B
c:\Users\Dell\anaconda3\envs\venv\lib\site-packages\IPython\core\history.py:783: size=2080 B, count=1, average=2080 B
c:\Users\Dell\anaconda3\envs\venv\lib\codeop.py:118: size=1624 B, count=19, average=85 B
c:\Users\Dell\anaconda3\envs\venv\lib\site-packages\ipykernel\iostream.py:288: size=1416 B, count=9, average=157 B
c:\Users\Dell\anaconda3\envs\venv\lib\site-packages\jupyter_client\session.py:98: size=1241 B, count=8, average=155 B
c